In [62]:
import pymupdf4llm
import re
import os
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langchain_chroma import Chroma
from dotenv import load_dotenv
load_dotenv()

True

In [25]:
file_path = "Attention.pdf"
markdown = pymupdf4llm.to_markdown(file_path)


=== Document parser messages ===
Using RapidOCR for OCR processing.
OCR on page.number=2/3.
OCR on page.number=3/4.


In [26]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ("####", "Header 4")
]

splitter = MarkdownHeaderTextSplitter(headers_to_split_on = headers_to_split_on)
markdown_chunks = splitter.split_text(markdown)

In [27]:
def additional_recursive_chunker(markdown_chunks : list[Document], chunk_size = 600, chunk_overlap = 100):

    good_chunk = []
    chunk_count = 0
    recursive_text_splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, 
                                                             chunk_overlap = chunk_overlap, 
                                                             separators=["\n\n", "\n", ". ", " ", ""])
    
    for chunk in markdown_chunks:
        if(len(chunk.page_content) > chunk_size):
            meta_data = chunk.metadata
            chunks = recursive_text_splitter.split_documents([chunk])
            for i in chunks:
                chunk_count = chunk_count + 1
                i.metadata["chunk_id"] = chunk_count
                good_chunk.append(i)
            chunk_count = 0
        else:
            good_chunk.append(chunk)
    return good_chunk

In [28]:
# this is the final chunk and metadata which is to be stores and embedded
final_chunks = additional_recursive_chunker(markdown_chunks)

In [29]:
embedding_model = OllamaEmbeddings(model="llama3.2")

In [30]:
# embed each of the query and then store them in the pgvector
vector_store = Chroma(
    collection_name= "my_docs",
    embedding_function=embedding_model,
    persist_directory="./store"
)

In [ ]:
ids = vector_store.add_documents(
    documents = final_chunks,)

In [ ]:
len(ids)

83

In [31]:
filter_condition = {}
search_kwargs = {}

In [32]:
if len(filter_condition) == 0:
    search_kwargs = {
        'k' : 3,
        "fetch_k" : 10,
        "lambda_mult" : 0.7,
    }
else:
    search_kwargs = {
        'k' : 3,
        "fetch_k" : 10,
        "lambda_mult" : 0.7,
        "filter" : {
            **filter_condition
        }
    },

In [33]:
search_kwargs

{'k': 3, 'fetch_k': 10, 'lambda_mult': 0.7}

In [34]:
retriver = vector_store.as_retriever(
    search_type = "mmr",
    search_kwargs = search_kwargs
)

In [35]:
ans =retriver.invoke("How does an attention mechanism work?")

In [36]:
ans

[Document(id='35e55b56-3c2d-4fb3-aded-3ecb083fe0c5', metadata={'Header 1': '**Attention Is All You Need**', 'Header 2': '**1 Introduction**', 'chunk_id': 7}, page_content='Attention mechanisms have become an integral part of compelling sequence modeling and transduction models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2, 16]. In all but a few cases [22], however, such attention mechanisms are used in conjunction with a recurrent network.'),
 Document(id='28bcbc81-a251-4209-a789-b6de8865539c', metadata={'Header 3': 'Ouput Probabilities', 'Header 2': '**3 Model Architecture**', 'Header 4': '**3.2.3 Applications of Attention in our Model**', 'chunk_id': 2, 'Header 1': '**Attention Is All You Need**'}, page_content='- The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each

In [37]:
for i in ans:
    print(i.page_content)
    print("=" * 150)

Attention mechanisms have become an integral part of compelling sequence modeling and transduction models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2, 16]. In all but a few cases [22], however, such attention mechanisms are used in conjunction with a recurrent network.
- The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each position in the encoder can attend to all positions in the previous layer of the encoder.
An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key.


In [38]:
for i in ans:
    print(i.metadata)

{'Header 1': '**Attention Is All You Need**', 'Header 2': '**1 Introduction**', 'chunk_id': 7}
{'Header 3': 'Ouput Probabilities', 'Header 2': '**3 Model Architecture**', 'Header 4': '**3.2.3 Applications of Attention in our Model**', 'chunk_id': 2, 'Header 1': '**Attention Is All You Need**'}
{'Header 3': 'Ouput Probabilities', 'Header 2': '**3 Model Architecture**', 'Header 1': '**Attention Is All You Need**', 'Header 4': '**3.2 Attention**'}


### Evaluation of the rag system without hybrid search and reranking

#### Evaluation LLMs

In [53]:
from openai import OpenAI
from groq import Groq
from ragas.llms import llm_factory

OllamaClient = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434"
)

GroqClient = Groq(
    api_key=os.getenv("api_key")
)

GroqEvaluation = llm_factory(
    model="llama-3.3-70b-versatile",
    client=GroqClient
)

OllamaEvaluation = llm_factory(
    "ollama",
    provider="openai",
    client=OllamaClient
)

c:\Users\Nitro\Desktop\productionReadyRAGapp\venv\Lib\site-packages\ragas\llms\base.py:556: UserWarning: Client should be an instance of openai.OpenAI or openai.AsyncOpenAI. Unexpected behavior may occur with other client types.
  return instructor.from_openai(client, mode=instructor.Mode.JSON)


#### RAG LLMs

In [63]:
OllamaLLM = ChatOllama(model="llama3.2")
GroqLLM = ChatGroq(
    api_key=os.getenv("api_key"),
    model="llama-3.3-70b-versatile"
)


#### Metrics

In [66]:
from ragas.metrics.collections import ( 
    ContextPrecision,
    ContextRecall, 
    ContextRelevance, 
    Faithfulness, 
    AnswerCorrectness, 
    AnswerRelevancy )


In [ ]:
# evaluation retriever
def recall_k():
    # -> how many of relevant chunks were retreived out of all the relevant chunks available
    pass

def precision_k():
    # -> how many of the retrieved chunks were relevant chunks to the the question

    pass

# evaluate the ranker (reranker or retriever's ranker)
def mrr():
    # -> the first relevant chunk out of all the retrieved chunk which are returned in order
    pass

def ndcg():
    # -> this metrics check whether the chunks are arranged in an descending order of chunks on 
    # the basis of relevancy
    pass

#### dataset for evaluation
this will contain the dataset which will contain:
- question
- relevant chunks (ground truth)
- reference of final answer (ground truth)



In [67]:
# this will contain the dataset which will contain

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

### RAG with hybrid search

In [16]:
# we need to perform the retrieval on the basis of the semantic and the keyword and the fuse then perform rrf on them to find the best chunk
# semantic search is the normal search for this


### BM25 as the keyword search

In [17]:
# keyword search is done using the bm25 algorithm

### Combining the chunks from dense and sparse search

In [18]:
# rrf

In [19]:
# reranker

### Generation of the answer from the llm

In [20]:
# generation

### Evaluate the chunks and answer after the changes in the pipeline

In [21]:
# evaluation criterias